# 05 - Results Summary

This notebook is the narrative dashboard for the project. It does not replace the detailed notebooks; it collects the main evidence, highlights which artefacts support each claim, and keeps methodological caveats visible.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.append(str(PROJECT_ROOT / "src"))

from config import OUTPUTS_FIGURES_DIR, OUTPUTS_TABLES_DIR

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)
plt.rcParams["figure.figsize"] = (8, 4)


## 1. Evidence Inventory

This table maps each major thesis result to the artefact that currently supports it. It is useful when editing the dissertation because it prevents claims from drifting away from the files we actually have.


In [ ]:
evidence_inventory = pd.DataFrame(
    [
        {"theme": "Segmentation choice", "main_file": "study2_windowing_selection_evidence.csv", "status": "recovered quantitative evidence; pragmatic choice"},
        {"theme": "Final baselines", "main_file": "final_baseline_summary.csv", "status": "final baseline table"},
        {"theme": "Feature selection", "main_file": "utility_feature_selection_summary.csv", "status": "exploratory single split"},
        {"theme": "Top-k trade-off", "main_file": "topk_utility_linkability_curve.csv", "status": "exploratory curve"},
        {"theme": "Feature removal", "main_file": "feature_removal_tradeoff_summary.csv", "status": "exploratory single split"},
        {"theme": "Privacy transformations", "main_file": "privacy_transform_full_final_summary.csv", "status": "final split-aware multi-seed study"},
        {"theme": "Fitted transform parameters", "main_file": "privacy_transform_fitted_parameters.csv", "status": "reproducibility artefact"},
        {"theme": "Final representation trade-off", "main_file": "final_representation_tradeoff_seed42.csv", "status": "broad representation comparison; seed 42"},
        {"theme": "Encoder+DP multi-seed", "main_file": "final_encoder_dp_multiseed_summary.csv", "status": "final multi-seed operational check"},
        {"theme": "Operational linkage", "main_file": "operational_linkage_final_summary.csv", "status": "synthetic gallery evaluation"},
        {"theme": "Computational protocol", "main_file": "model_computational_protocol_summary.csv", "status": "implementation reproducibility artefact"},
    ]
)
evidence_inventory


## 2. Load Main Result Tables

The rest of the notebook reads only compact result tables, not the full ECG feature dataset.


In [ ]:
segmentation_selection_df = pd.read_csv(OUTPUTS_TABLES_DIR / "segmentation_selection_summary.csv")
baseline_results_df = pd.read_csv(OUTPUTS_TABLES_DIR / "final_baseline_summary.csv")
same_data_tradeoff_df = pd.read_csv(OUTPUTS_TABLES_DIR / "same_data_tradeoff_summary.csv")
feature_removal_tradeoff_df = pd.read_csv(OUTPUTS_TABLES_DIR / "feature_removal_tradeoff_summary.csv")
privacy_transformations_df = pd.read_csv(OUTPUTS_TABLES_DIR / "privacy_transformations_summary.csv")
pca_components_df = pd.read_csv(OUTPUTS_TABLES_DIR / "pca_components_tradeoff_curve.csv")
operational_linkage_summary_df = pd.read_csv(OUTPUTS_TABLES_DIR / "operational_linkage_final_summary.csv")
model_protocol_df = pd.read_csv(OUTPUTS_TABLES_DIR / "model_computational_protocol_summary.csv")
final_representation_tradeoff_df = pd.read_csv(OUTPUTS_TABLES_DIR / "final_representation_tradeoff_seed42.csv")
final_encoder_dp_multiseed_df = pd.read_csv(OUTPUTS_TABLES_DIR / "final_encoder_dp_multiseed_summary.csv")
final_thesis_ready_df = pd.read_csv(OUTPUTS_TABLES_DIR / "final_thesis_ready_representation_table.csv")


## 3. Segmentation and Baseline Summary

These are the core setup results: which windowing configuration was retained and which models are used as baselines.


In [ ]:
display(segmentation_selection_df)
display(baseline_results_df)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(segmentation_selection_df["config"], segmentation_selection_df["utility_logreg_f1"], color="#457b9d")
axes[0].set_title("Utility F1 by segmentation candidate")
axes[0].set_ylabel("F1")
axes[1].bar(segmentation_selection_df["config"], segmentation_selection_df["linkability_xgb_roc_auc"], color="#d62828")
axes[1].set_title("Linkability ROC-AUC by segmentation candidate")
axes[1].set_ylabel("ROC-AUC")
plt.tight_layout()
plt.show()


## 4. Feature Trade-off Snapshot

This view puts the full-feature baseline, top-150 subset and progressive removal experiment side by side.


In [ ]:
display(same_data_tradeoff_df)
display(feature_removal_tradeoff_df)


In [ ]:
fig, ax1 = plt.subplots(figsize=(8, 4))
ax2 = ax1.twinx()
ax1.plot(feature_removal_tradeoff_df["n_removed"], feature_removal_tradeoff_df["utility_f1"], marker="o", color="#457b9d", label="Utility F1")
ax2.plot(feature_removal_tradeoff_df["n_removed"], feature_removal_tradeoff_df["linkability_roc_auc"], marker="s", color="#d62828", label="Linkability ROC-AUC")
ax1.set_xlabel("Removed features")
ax1.set_ylabel("Utility F1", color="#457b9d")
ax2.set_ylabel("Linkability ROC-AUC", color="#d62828")
ax1.set_title("Feature removal: utility vs linkability")
lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="best")
plt.tight_layout()
plt.show()


## 5. Transformation Trade-off Snapshot

The final transformation results are multi-seed and split-aware. This is the strongest evidence for the transformation part of Study II.


In [ ]:
privacy_transformations_df.sort_values("delta_linkability_roc_auc")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(privacy_transformations_df["linkability_roc_auc"], privacy_transformations_df["utility_f1"], s=90, color="#4c78a8")
for _, row in privacy_transformations_df.iterrows():
    ax.annotate(row["transform_name"], (row["linkability_roc_auc"], row["utility_f1"]), xytext=(5, 4), textcoords="offset points")
ax.set_xlabel("Linkability ROC-AUC")
ax.set_ylabel("Utility F1")
ax.set_title("Privacy transformation trade-off")
plt.tight_layout()
plt.show()


## 6. Operational Linkage Snapshot

This section keeps the distinction between balanced pairwise separability and lower-prevalence gallery-style evaluation visible.


In [ ]:
operational_linkage_summary_df[[
    "gallery_negatives",
    "positive_prevalence_mean",
    "roc_auc_mean",
    "pr_auc_mean",
    "tpr_at_fpr_0.001_mean",
    "recall_at_1_mean",
    "recall_at_5_mean",
    "mrr_mean",
]]


## 7. Final Representation Trade-off

This is the thesis-facing synthesis: broad representation families are compared in the seed-42 representation sweep, while the strongest encoder+DP candidates are summarised across three patient-level splits.


In [ ]:
key_representations = [
    "raw 208 features",
    "PCA 64",
    "Random projection 64",
    "Supervised encoder 8",
    "Encoder dim8 + DP eps50 clip2",
    "Encoder dim8 + DP eps50 clip4",
]
final_representation_tradeoff_df[
    final_representation_tradeoff_df["representation"].isin(key_representations)
][[
    "representation",
    "dimension",
    "utility_balanced_accuracy",
    "pairwise_linkability_roc_auc",
    "operational_pr_auc",
    "operational_recall_at_1",
]]


In [ ]:
final_encoder_dp_multiseed_df[[
    "representation",
    "utility_balanced_accuracy_mean",
    "utility_balanced_accuracy_std",
    "pairwise_linkability_roc_auc_mean",
    "pairwise_linkability_roc_auc_std",
    "pairwise_linkability_eer_mean",
    "pairwise_linkability_eer_std",
    "link_operational_pr_auc_mean",
    "link_operational_pr_auc_std",
    "link_operational_eer_mean",
    "link_operational_eer_std",
    "recall_at_1_mean",
    "recall_at_1_std",
    "mrr_mean",
    "mrr_std",
]]


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
plot_df = final_encoder_dp_multiseed_df.copy()
axes[0].bar(plot_df["representation"], plot_df["link_operational_pr_auc_mean"], yerr=plot_df["link_operational_pr_auc_std"], color="#457b9d", capsize=3)
axes[0].set_title("Operational PR-AUC, 0.1% prevalence")
axes[0].set_ylabel("PR-AUC")
axes[0].tick_params(axis="x", rotation=25)
axes[0].grid(axis="y", alpha=0.25)
axes[1].bar(plot_df["representation"], plot_df["recall_at_1_mean"], yerr=plot_df["recall_at_1_std"], color="#e76f51", capsize=3)
axes[1].set_title("Recall@1, 0.1% prevalence")
axes[1].set_ylabel("Recall@1")
axes[1].tick_params(axis="x", rotation=25)
axes[1].grid(axis="y", alpha=0.25)
plt.tight_layout()
figure_path = OUTPUTS_FIGURES_DIR / "final_encoder_dp_operational_multiseed.png"
fig.savefig(figure_path, dpi=200, bbox_inches="tight")
plt.show()
figure_path


### Thesis-Ready Final Table

This compact table is the one intended for dissertation writing. It separates exploratory seed-42 broad representation evidence from the final three-seed encoder+DP operating points.


In [ ]:
final_thesis_ready_df


## 8. Computational Protocol Snapshot

This is the quick reference for reproducibility: split protocol, folds, preprocessing, imbalance handling, thresholds and seeds.


In [ ]:
model_protocol_df[["protocol_scope", "task", "models", "folds", "feature_preprocessing", "imbalance_handling", "seeds"]]


## Current Conclusions

- The final segmentation is `w2_o0p5`, but it should be framed as a pragmatic operating point rather than an optimized/Pareto-proven choice.
- Utility is evaluated as AF/non-AF segment classification; linkability is same-retained-identifier pair separability, with an additional synthetic gallery check.
- The utility baseline is reasonable but not perfect; record-level aggregation improves utility substantially.
- Linkability remains very high under pairwise evaluation, even with harder sampling.
- PCA and random projection are the only transformations that materially reduce linkability in the final split-aware study; random projection has the larger utility cost.
- Top-150 and progressive-removal results are useful for story-building and trade-off intuition, but remain exploratory single-split analyses.

- The final representation-level result is strongest for the supervised bottleneck encoder plus DP-calibrated embedding perturbation: it keeps utility in the target range while substantially reducing operational linkage metrics.
- `eps50/clip2` is the best balanced operating point; `eps50/clip4` is a stronger privacy point with a larger but still moderate utility cost.
- EER was added as a complementary leakage metric; higher EER indicates weaker identity discrimination.
- DP-calibrated embedding perturbation should not be described as DP-SGD or end-to-end differentially private training.
